# Korxona xodimlari HR tahlili

**Muallif:** Xusniddin Yergashov | **Maqsad:** Junior Data Analyst

**Loyiha haqida:** 22 000 xodimdan iborat korxona ma'lumotini API orqali yuklab, pandas yordamida tozaladim va tahlil qildim.

**Asosiy savollar:**
1. Qaysi bo'limlarda maosh eng yuqori?
2. Ma'lumot darajasi va staj maoshga qanchalik ta'sir qiladi?
3. KPI ball bonus bilan bog'liqmi?
4. Ayol va erkak xodimlar maoshida farq bormi?

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

url = "https://dataapi-nch9.onrender.com/korxona"
data = requests.get(url, timeout=60).json()
df = pd.DataFrame(data)
print(df.shape)
df.head()

## 1. Ma'lumotni tekshirish va tozalash

In [ ]:
df.info()
print("\nBo'sh qiymatlar:\n", df.isna().sum())
print("\nTakroriy xodim_id:", df['xodim_id'].duplicated().sum())
print("\nIsh holati turlari:\n", df['ish_holati'].value_counts())

In [ ]:
# Sana ustunini datetime ga o'tkazish
df['ishga_kirgan_sana'] = pd.to_datetime(df['ishga_kirgan_sana'])

# Jami daromad (oylik + bonus)
df['jami_daromad'] = df['oylik'] + df['bonus']

# Yosh guruhlari
df['yosh_guruh'] = pd.cut(df['yosh'], bins=[17, 25, 35, 45, 55, 100],
                          labels=['18-25', '26-35', '36-45', '46-55', '56+'])
df.describe()

## 2. Umumiy ko'rsatkichlar

In [ ]:
print(f"Jami xodimlar: {len(df):,}")
print(f"Jami oylik fondi: {df['oylik'].sum():,.0f} so'm")
print(f"O'rtacha oylik: {df['oylik'].mean():,.0f} so'm")
print(f"Median oylik: {df['oylik'].median():,.0f} so'm")
print(f"Eng yuqori / eng past: {df['oylik'].max():,.0f} / {df['oylik'].min():,.0f} so'm")
print(f"Jami bonus: {df['bonus'].sum():,.0f} so'm")
print()
print(df['jins'].value_counts())
print()
print(df['ish_holati'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

## 3. Bo'limlar bo'yicha tahlil

In [ ]:
bolim = df.groupby('bolim').agg(
    xodimlar=('xodim_id', 'count'),
    ortacha_oylik=('oylik', 'mean'),
    ortacha_kpi=('kpi_ball', 'mean')
).sort_values('ortacha_oylik').round(1)
display(bolim)

fig, ax = plt.subplots()
ax.barh(bolim.index, bolim['ortacha_oylik'] / 1e6, color='#3b6fb6')
ax.set_xlabel("O'rtacha oylik (mln so'm)")
ax.set_title("Bo'limlar bo'yicha o'rtacha oylik")
for i, v in enumerate(bolim['ortacha_oylik'] / 1e6):
    ax.text(v + 0.05, i, f"{v:.1f}", va='center')
plt.tight_layout()
plt.savefig('grafik_bolim.png', dpi=150)
plt.show()

## 4. Ma'lumot darajasi va staj maoshga ta'siri

In [ ]:
tartib = ['O\'rta', 'Bakalavr', 'Magistr', 'PhD']
mal = df.groupby('malumoti')['oylik'].mean().sort_values()
print(mal.round(0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(mal.index, mal.values / 1e6, color='#2a9d8f')
axes[0].set_title("Ma'lumot darajasi bo'yicha o'rtacha oylik")
axes[0].set_ylabel("mln so'm")

df['staj_guruh'] = pd.cut(df['staj_yil'], bins=[-1, 2, 5, 10, 20, 60],
                          labels=['0-2', '3-5', '6-10', '11-20', '20+'])
st = df.groupby('staj_guruh', observed=True)['oylik'].mean()
axes[1].plot(st.index.astype(str), st.values / 1e6, marker='o', color='#e76f51')
axes[1].set_title("Staj bo'yicha o'rtacha oylik")
axes[1].set_xlabel("Staj (yil)")
axes[1].set_ylabel("mln so'm")

plt.tight_layout()
plt.savefig('grafik_malumot_staj.png', dpi=150)
plt.show()

print("Staj va oylik korrelyatsiyasi:", round(df['staj_yil'].corr(df['oylik']), 2))

## 5. KPI va bonus

In [ ]:
print("KPI va bonus korrelyatsiyasi:", round(df['kpi_ball'].corr(df['bonus']), 2))
print("KPI va oylik korrelyatsiyasi:", round(df['kpi_ball'].corr(df['oylik']), 2))

yuqori = df[df['kpi_ball'] >= 90]
print(f"\nKPI >= 90 bo'lgan xodimlar: {len(yuqori):,} ({len(yuqori)/len(df)*100:.1f}%)")
print(f"Ularning jami bonusi: {yuqori['bonus'].sum():,.0f} so'm")

df['kpi_guruh'] = pd.cut(df['kpi_ball'], bins=[0, 60, 70, 80, 90, 100],
                         labels=['<60', '60-70', '70-80', '80-90', '90+'])
kb = df.groupby('kpi_guruh', observed=True)['bonus'].mean()

fig, ax = plt.subplots()
ax.bar(kb.index.astype(str), kb.values / 1e6, color='#8d6bb8')
ax.set_title("KPI guruhlari bo'yicha o'rtacha bonus")
ax.set_xlabel("KPI ball")
ax.set_ylabel("mln so'm")
plt.tight_layout()
plt.savefig('grafik_kpi_bonus.png', dpi=150)
plt.show()

## 6. Jins bo'yicha maosh

In [ ]:
jins = df.groupby('jins')['oylik'].agg(['count', 'mean', 'median']).round(0)
display(jins)

pivot = df.pivot_table(index='bolim', columns='jins', values='oylik', aggfunc='mean') / 1e6
pivot.plot(kind='bar', figsize=(11, 5), color=['#e07a9f', '#4a7fb5'])
plt.title("Bo'limlar va jins bo'yicha o'rtacha oylik")
plt.ylabel("mln so'm")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('grafik_jins.png', dpi=150)
plt.show()

## 7. Viloyatlar bo'yicha taqsimot

In [ ]:
vil = df.groupby('viloyat').agg(xodimlar=('xodim_id', 'count'),
                                ortacha_oylik=('oylik', 'mean')).sort_values('xodimlar', ascending=False).round(0)
display(vil)

## Power BI uchun tayyor ma'lumot
Tozalangan jadvalni CSV qilib saqlaymiz. Power BI dashboardi shu fayldan quriladi.

In [ ]:
export = df.drop(columns=['yosh_guruh', 'staj_guruh', 'kpi_guruh'], errors='ignore')
export.to_csv('xodimlar_tozalangan.csv', index=False, encoding='utf-8-sig')
print(export.shape, "-> xodimlar_tozalangan.csv saqlandi")

## 8. Asosiy xulosalar

> Kodni ishga tushirgach, yuqoridagi natijalarga qarab shu bo'limni o'z so'zlaringiz bilan to'ldiring.

1. **Bo'limlar:** eng yuqori maosh ... bo'limida (... mln so'm), eng past ... bo'limida.
2. **Ma'lumot va staj:** ... (korrelyatsiya ...).
3. **KPI va bonus:** ... 
4. **Jins:** ayol va erkak o'rtasidagi o'rtacha maosh farqi ...
5. **Tavsiya:** ...

## Foydalanilgan vositalar
Python, pandas, NumPy, matplotlib, requests